In [4]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Data Preprocessing

In [2]:
import pandas as pd
import json

project_dir = '/content/drive/MyDrive/VisualReview'

print("Loading reviews...")
reviews = []
with open(f'{project_dir}/Appliances.jsonl/Appliances.jsonl') as f:
    for line in f:
        reviews.append(json.loads(line))
reviews_df = pd.DataFrame(reviews)
print(f"Reviews loaded: {len(reviews_df):,}")

print("Loading metadata...")
meta = []
with open(f'{project_dir}/meta_Appliances.jsonl/meta_Appliances.jsonl') as f:
    for line in f:
        meta.append(json.loads(line))
meta_df = pd.DataFrame(meta)
print(f"Meta loaded: {len(meta_df):,}")

Loading reviews...
Reviews loaded: 2,128,605
Loading metadata...
Meta loaded: 94,327


In [3]:
# Cell 2: Inspect both dataframes
print("=== REVIEWS COLUMNS ===")
print(reviews_df.columns.tolist())
print("\n=== REVIEWS SAMPLE ===")
print(reviews_df.head(2))

print("\n=== META COLUMNS ===")
print(meta_df.columns.tolist())
print("\n=== META SAMPLE ===")
print(meta_df.head(2))

=== REVIEWS COLUMNS ===
['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']

=== REVIEWS SAMPLE ===
   rating              title                                   text images  \
0     5.0         Work great  work great. use a new one every month     []   
1     5.0  excellent product                Little on the thin side     []   

         asin parent_asin                       user_id      timestamp  \
0  B01N0TQ0OH  B01N0TQ0OH  AGKHLEW2SOWHNMFQIJGBECAF7INQ  1519317108692   
1  B07DD2DMXB  B07DD37QPZ  AHWWLSPCJMALVHDDVSUGICL6RUCA  1664746863446   

   helpful_vote  verified_purchase  
0             0               True  
1             0               True  

=== META COLUMNS ===
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author']

=== META SAMPLE ===
          

In [4]:
# Cell 3 (FIXED): Keep only columns we need
reviews_clean = reviews_df[[
    'parent_asin',
    'rating',
    'text',
]].copy()

meta_clean = meta_df[[
    'parent_asin',
    'title',
    'images',
    'main_category',
]].copy()

print("Reviews shape:", reviews_clean.shape)
print("Meta shape:", meta_clean.shape)

Reviews shape: (2128605, 3)
Meta shape: (94327, 4)


In [5]:
# Cell 4 (FIXED): Extract image URL — images is a LIST of dicts
def extract_image_url(images):
    try:
        if not images or len(images) == 0:
            return None
        # each item is a dict with thumb, large, hi_res, variant
        # find the MAIN image first, fall back to first available
        for img in images:
            if img.get('variant') == 'MAIN':
                return img.get('hi_res') or img.get('large') or None
        # fallback: just return first image's hi_res or large
        first = images[0]
        return first.get('hi_res') or first.get('large') or None
    except:
        return None

meta_clean = meta_clean.copy()
meta_clean['image_url'] = meta_clean['images'].apply(extract_image_url)

print("Products with valid image URL:", meta_clean['image_url'].notna().sum())
print("Products without image URL:", meta_clean['image_url'].isna().sum())
print("\nSample image URL:", meta_clean['image_url'].dropna().iloc[0])

Products with valid image URL: 94326
Products without image URL: 1

Sample image URL: https://m.media-amazon.com/images/I/61zNIJh6ZCL._SL1500_.jpg


In [6]:
# Cell 5 (FIXED): Join reviews with metadata
merged_df = reviews_clean.merge(
    meta_clean[['parent_asin', 'image_url', 'title']],
    on='parent_asin',
    how='inner'
)

print("Merged shape:", merged_df.shape)
print("\nSample row:")
print(merged_df.iloc[0])

Merged shape: (2128605, 5)

Sample row:
parent_asin                                           B01N0TQ0OH
rating                                                       5.0
text                       work great. use a new one every month
image_url      https://m.media-amazon.com/images/I/71zqD75W03...
title          Geesta 12-Pack Premium Activated Charcoal Wate...
Name: 0, dtype: object


In [7]:
# Cell 6 (FIXED): Filter low quality records
print("Before filtering:", f"{len(merged_df):,}")

filtered_df = merged_df[
    merged_df['image_url'].notna() &
    merged_df['text'].notna() &
    (merged_df['text'].str.len() >= 50) &
    (merged_df['text'].str.len() <= 1000) &
    merged_df['rating'].notna()
].copy()

print("After filtering:", f"{len(filtered_df):,}")
print("\nRating distribution:")
print(filtered_df['rating'].value_counts().sort_index())

Before filtering: 2,128,605
After filtering: 1,391,466

Rating distribution:
rating
1.0    201596
2.0     66238
3.0     83407
4.0    144849
5.0    895376
Name: count, dtype: int64


In [8]:
# Cell 7: Keep products with at least 3 reviews
reviews_per_product = filtered_df.groupby('parent_asin')['rating'].agg(
    count='count',
    unique_ratings=lambda x: x.nunique()
).reset_index()

good_products = reviews_per_product[reviews_per_product['count'] >= 3]['parent_asin']
filtered_df = filtered_df[filtered_df['parent_asin'].isin(good_products)].copy()

print(f"Products with 3+ reviews: {len(good_products):,}")
print(f"Total records remaining: {len(filtered_df):,}")

Products with 3+ reviews: 38,927
Total records remaining: 1,340,320


In [9]:
# Cell 8: Save
import os
os.makedirs(f'{project_dir}/processed', exist_ok=True)

output_path = f'{project_dir}/processed/Appliances_joined.jsonl'
filtered_df.to_json(output_path, orient='records', lines=True)
print(f"Saved to {output_path}")
print(f"Final dataset: {len(filtered_df):,} review-image pairs")

Saved to /content/drive/MyDrive/VisualReview/processed/Appliances_joined.jsonl
Final dataset: 1,340,320 review-image pairs


# Image download through URLs

In [10]:
# Cell 1: Check how many unique products we need to download images for
unique_products = filtered_df[['parent_asin', 'image_url']].drop_duplicates(subset='parent_asin')
print(f"Unique products to download images for: {len(unique_products):,}")
print("\nSample URLs:")
print(unique_products['image_url'].head(5).tolist())

Unique products to download images for: 38,927

Sample URLs:
['https://m.media-amazon.com/images/I/61fC-rtcH7L._AC_SL1500_.jpg', 'https://m.media-amazon.com/images/I/614VU14HAdL._AC_SL1200_.jpg', 'https://m.media-amazon.com/images/I/61S2sHE28UL._SL1500_.jpg', 'https://m.media-amazon.com/images/I/71meZeL4C5L._AC_SL1500_.jpg', 'https://m.media-amazon.com/images/I/61dbzuTcrRL._AC_SL1500_.jpg']


In [11]:
# Cell 2: Create image directory
import os

image_dir = f'{project_dir}/data/images'
os.makedirs(image_dir, exist_ok=True)
print("Image directory created at:", image_dir)

Image directory created at: /content/drive/MyDrive/VisualReview/data/images


In [12]:
# Cell 3 (FIXED): Download images with parallel processing + progress bar
import requests
import time
import os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

def download_image(args):
    """Download a single image and save as {asin}.jpg"""
    asin, url, save_dir = args
    save_path = Path(save_dir) / f"{asin}.jpg"

    if save_path.exists():
        return asin, 'skipped'

    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            with open(save_path, 'wb') as f:
                f.write(response.content)
            return asin, 'success'
        else:
            return asin, f'failed_{response.status_code}'
    except Exception as e:
        return asin, f'error_{str(e)[:30]}'

# Prepare arguments
args_list = [
    (row['parent_asin'], row['image_url'], image_dir)
    for _, row in unique_products.iterrows()
]

results = {'success': 0, 'skipped': 0, 'failed': 0}
failed_asins = []

# 16 workers is a good balance — fast but won't get rate limited
NUM_WORKERS = 16

print(f"Downloading {len(args_list):,} images with {NUM_WORKERS} parallel workers...\n")

with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    futures = {executor.submit(download_image, args): args for args in args_list}

    with tqdm(total=len(args_list), desc="Downloading images", unit="img") as pbar:
        for future in as_completed(futures):
            asin, status = future.result()

            if status == 'success':
                results['success'] += 1
            elif status == 'skipped':
                results['skipped'] += 1
            else:
                results['failed'] += 1
                failed_asins.append(asin)

            # Update progress bar with live stats
            pbar.set_postfix({
                'success': results['success'],
                'skipped': results['skipped'],
                'failed': results['failed']
            })
            pbar.update(1)

print("\n=== DOWNLOAD COMPLETE ===")
print(f"Successfully downloaded : {results['success']:,}")
print(f"Skipped (already existed): {results['skipped']:,}")
print(f"Failed                  : {results['failed']:,}")


=== DOWNLOAD COMPLETE ===
Successfully downloaded : 38,899
Skipped (already existed): 0
Failed                  : 28


In [13]:
# Cell 4 (FIXED): Validate images in parallel + progress bar
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

def validate_image(asin):
    path = f"{image_dir}/{asin}.jpg"
    try:
        img = Image.open(path)
        img.verify()
        return asin, 'valid'
    except Exception:
        return asin, 'corrupted'

asins_to_check = unique_products['parent_asin'].tolist()
corrupted = []
valid = 0

print(f"Validating {len(asins_to_check):,} images...\n")

with ThreadPoolExecutor(max_workers=16) as executor:
    futures = {executor.submit(validate_image, asin): asin for asin in asins_to_check}

    with tqdm(total=len(asins_to_check), desc="Validating images", unit="img") as pbar:
        for future in as_completed(futures):
            asin, status = future.result()
            if status == 'valid':
                valid += 1
            else:
                corrupted.append(asin)
            pbar.update(1)

print(f"\nValid images    : {valid:,}")
print(f"Corrupted/missing: {len(corrupted):,}")

Validating 38,927 images...



Validating images:   0%|          | 0/38927 [00:00<?, ?img/s]


Valid images    : 38,899
Corrupted/missing: 28


In [14]:
# Cell 5: Remove reviews for products with failed/corrupted images
# so our final dataset only has rows where the image actually exists

good_asins = set(unique_products['parent_asin']) - set(corrupted) - set(failed_asins)

clean_df = filtered_df[filtered_df['parent_asin'].isin(good_asins)].copy()

# Add local image path column
clean_df['image_path'] = clean_df['parent_asin'].apply(
    lambda x: f"{image_dir}/{x}.jpg"
)

print(f"Final clean dataset: {len(clean_df):,} reviews")
print(f"Unique products: {clean_df['parent_asin'].nunique():,}")
print(f"\nSample row:")
print(clean_df.iloc[0])

Final clean dataset: 1,339,957 reviews
Unique products: 38,899

Sample row:
parent_asin                                           B078W2BJY8
rating                                                       5.0
text           I wasn't sure whether these were worth it or n...
image_url      https://m.media-amazon.com/images/I/61fC-rtcH7...
title          Filterlogic UKF8001 Water Filter, Replacement ...
image_path     /content/drive/MyDrive/VisualReview/data/image...
Name: 3, dtype: object


In [15]:
# Cell 6: Save the final clean dataset
output_path = f'{project_dir}/processed/Appliances_final.jsonl'
clean_df.to_json(output_path, orient='records', lines=True)
print(f"Saved: {output_path}")
print(f"Total reviews: {len(clean_df):,}")
print(f"Total products with images: {clean_df['parent_asin'].nunique():,}")

Saved: /content/drive/MyDrive/VisualReview/processed/Appliances_final.jsonl
Total reviews: 1,339,957
Total products with images: 38,899


# Tokenizing the text using BPE

In [5]:
# Cell 1: Load the final cleaned dataset
import pandas as pd
import json

project_dir = '/content/drive/MyDrive/VisualReview'

print("Loading final dataset...")
df = pd.read_json(f'{project_dir}/processed/Appliances_final.jsonl', lines=True)
print(f"Loaded: {len(df):,} reviews")
print(f"Rating distribution:\n{df['rating'].value_counts().sort_index()}")

Loading final dataset...
Loaded: 1,339,957 reviews
Rating distribution:
rating
1    191533
2     63526
3     79886
4    139713
5    865299
Name: count, dtype: int64


In [12]:
# Cell 2 : Hard cap per rating class
per_class = 60_000  # 5 classes × 60K = 300K total
# (keeps 1-star which only has ~57K, caps the rest)

sampled_df = df.groupby('rating', group_keys=False).apply(
    lambda x: x.sample(n=min(len(x), per_class), random_state=42),
    include_groups=True
).reset_index(drop=True)

print(f"Sampled dataset size: {len(sampled_df):,}")
print(f"\nRating distribution after sampling:")
print(sampled_df['rating'].value_counts().sort_index())
print(f"\nRating percentages:")
print((sampled_df['rating'].value_counts().sort_index() / len(sampled_df) * 100).round(2))

/tmp/ipykernel_61674/1841747212.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_df = df.groupby('rating', group_keys=False).apply(


Sampled dataset size: 300,000

Rating distribution after sampling:
rating
1    60000
2    60000
3    60000
4    60000
5    60000
Name: count, dtype: int64

Rating percentages:
rating
1    20.0
2    20.0
3    20.0
4    20.0
5    20.0
Name: count, dtype: float64


In [13]:
# Cell 3: Save the sampled subset
output_path = f'{project_dir}/processed/Appliances_400k.jsonl'
sampled_df.to_json(output_path, orient='records', lines=True)
print(f"Saved: {output_path}")
print(f"Total reviews: {len(sampled_df):,}")
print(f"Unique products: {sampled_df['parent_asin'].nunique():,}")

Saved: /content/drive/MyDrive/VisualReview/processed/Appliances_400k.jsonl
Total reviews: 300,000
Unique products: 31,824


In [14]:
# Cell 5: Train BPE tokenizer from scratch on review text
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing

# Step 1: Initialize a blank BPE tokenizer
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

# Step 2: Define special tokens
# [UNK]  — unknown token for words not in vocabulary
# [PAD]  — padding token to make batches equal length
# [BOS]  — beginning of sequence
# [EOS]  — end of sequence
# [S1]-[S5] — sentiment control tokens (star ratings)
# [P1]-[P3] — persona control tokens (budget, casual, enthusiast)
# [A1]-[A5] — aspect control tokens (performance, design, value, durability, ease of use)

special_tokens = [
    "[UNK]", "[PAD]", "[BOS]", "[EOS]",
    "[S1]", "[S2]", "[S3]", "[S4]", "[S5]",   # sentiment
    "[P1]", "[P2]", "[P3]",                     # persona
    "[A1]", "[A2]", "[A3]", "[A4]", "[A5]",    # aspect
]

# Step 3: Configure the trainer
trainer = BpeTrainer(
    vocab_size=16000,          # vocabulary size — enough for review text
    min_frequency=3,           # ignore tokens appearing fewer than 3 times
    special_tokens=special_tokens,
    show_progress=True
)

# Step 4: Feed all review text to the trainer
# We write reviews to a temp txt file — that's what the trainer expects
import tempfile
import os

print("Writing reviews to temp file for tokenizer training...")
tmp_path = '/tmp/reviews_corpus.txt'
with open(tmp_path, 'w', encoding='utf-8') as f:
    for text in sampled_df['text'].dropna():
        f.write(text.strip() + '\n')

print(f"Corpus file written. Training BPE tokenizer on {len(sampled_df):,} reviews...")

# Step 5: Train
tokenizer.train(files=[tmp_path], trainer=trainer)
print("Tokenizer training complete!")
print(f"Vocabulary size: {tokenizer.get_vocab_size():,}")

Writing reviews to temp file for tokenizer training...
Corpus file written. Training BPE tokenizer on 300,000 reviews...
Tokenizer training complete!
Vocabulary size: 16,000


In [15]:
# Cell 6: Add BOS/EOS post-processing so every encoded sequence
# automatically gets [BOS] prepended and [EOS] appended
tokenizer.post_processor = TemplateProcessing(
    single="[BOS] $A [EOS]",
    special_tokens=[
        ("[BOS]", tokenizer.token_to_id("[BOS]")),
        ("[EOS]", tokenizer.token_to_id("[EOS]")),
    ],
)

# Quick sanity check
sample_text = "This product works great, very easy to install and use!"
encoded = tokenizer.encode(sample_text)
print("=== TOKENIZER SANITY CHECK ===")
print(f"Input text  : {sample_text}")
print(f"Tokens      : {encoded.tokens}")
print(f"Token IDs   : {encoded.ids}")
print(f"Decoded back: {tokenizer.decode(encoded.ids)}")

=== TOKENIZER SANITY CHECK ===
Input text  : This product works great, very easy to install and use!
Tokens      : ['[BOS]', 'This', 'product', 'works', 'great', ',', 'very', 'easy', 'to', 'install', 'and', 'use', '!', '[EOS]']
Token IDs   : [2, 600, 615, 702, 638, 30, 572, 705, 463, 614, 473, 556, 19, 3]
Decoded back: This product works great , very easy to install and use !


In [16]:
# Cell 7: Save the tokenizer to Drive
tokenizer_path = f'{project_dir}/tokenizer/appliances_bpe_tokenizer.json'
os.makedirs(f'{project_dir}/tokenizer', exist_ok=True)
tokenizer.save(tokenizer_path)
print(f"Tokenizer saved to: {tokenizer_path}")

# Verify it reloads correctly
from tokenizers import Tokenizer
reloaded = Tokenizer.from_file(tokenizer_path)
print(f"Reloaded vocabulary size: {reloaded.get_vocab_size():,}")
print("Tokenizer ready!")

Tokenizer saved to: /content/drive/MyDrive/VisualReview/tokenizer/appliances_bpe_tokenizer.json
Reloaded vocabulary size: 16,000
Tokenizer ready!
